# MIRACL / Meta-HMORL on OR-Gym-style Simple Supply Chain

This notebook is a cleaned version of `metah_s2_5000ts_orgym.ipynb` that uses `simple_state.py` instead of the Messiah-based `SimpleSC` / `TestSimpleSC` imports.  
It supports:

- RLlib/MAML meta-training with the OR-Gym-style `SimpleState` environment.
- Stable-Baselines3 PPO fine-tuning on the fixed-demand `TestSimpleState` environment.
- Native evaluation using the `info["mo_reward"]` and `info["mo_reward_raw"]` fields returned by `simple_state.py`.

Place `simple_state.py` in the same directory as this notebook, or set `ORGym_ENV_DIR` below.

In [23]:
# ---------------------------------------------------------------------
# 0. Paths and runtime settings
# ---------------------------------------------------------------------
import os
import sys
from pathlib import Path

# Main project directory containing algo_ray, utils, etc.
MIRACL_BASE_DIR = Path(os.environ.get("MIRACL_BASE_DIR", "/home/rifnyrachman7/_metamorl"))

# Directory containing simple_state.py.  In ChatGPT sandbox this is /mnt/data.
ORGYM_ENV_DIR = Path(os.environ.get("ORGYM_ENV_DIR", "/mnt/data"))

for p in [MIRACL_BASE_DIR, ORGYM_ENV_DIR, Path.cwd()]:
    p = Path(p)
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Keep PYTHONPATH additive instead of overwriting it.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(MIRACL_BASE_DIR), str(ORGYM_ENV_DIR), os.environ.get("PYTHONPATH", "")]
)

# Output directories. Change these if needed.
EXP_NAME = "metah_s2_orgym_5000ts"
CKPT_DIR = MIRACL_BASE_DIR / "mo_train" / EXP_NAME
FT_DIR = MIRACL_BASE_DIR / "fine_tuning" / EXP_NAME
PERF_DIR = FT_DIR / "performance"
for d in [CKPT_DIR, FT_DIR, PERF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NUM_PERIODS = 100
ROLLOUT_FRAGMENT_LENGTH = 32
TOTAL_STEPS = 5000
RECORD_EVERY = 250
EVAL_EPISODES = 50
RUNS = 10

print("MIRACL_BASE_DIR:", MIRACL_BASE_DIR)
print("ORGYM_ENV_DIR:", ORGYM_ENV_DIR)
print("CKPT_DIR:", CKPT_DIR)
print("FT_DIR:", FT_DIR)

MIRACL_BASE_DIR: /home/rifnyrachman7/_metamorl
ORGYM_ENV_DIR: /mnt/data
CKPT_DIR: /home/rifnyrachman7/_metamorl/mo_train/metah_s2_orgym_5000ts
FT_DIR: /home/rifnyrachman7/_metamorl/fine_tuning/metah_s2_orgym_5000ts


In [24]:
# ---------------------------------------------------------------------
# 1. Imports
# ---------------------------------------------------------------------
import time
import copy
import warnings
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pygmo as pg
except Exception as exc:
    pg = None
    warnings.warn(f"pygmo is unavailable; hypervolume cells will not run until installed. Error: {exc}")

import gymnasium as gym
from gymnasium.wrappers import TimeLimit

import ray
from ray import air, tune
from ray.air import Checkpoint
from ray.tune.registry import register_env

from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.monitor import Monitor

# Local MIRACL/RLlib algorithm.
from algo_ray.rllib.algorithms.maml.maml_psa import MAMLPsa, MAMLConfigPsa

# OR-Gym-style simple SC environment.
from simple_state import make_env, SimpleState, TestSimpleState

print("Imports loaded")

Imports loaded


In [25]:
# ---------------------------------------------------------------------
# 2. Simple-state smoke test
# ---------------------------------------------------------------------
train_env_smoke = make_env({"fixed_demand": False, "num_periods": NUM_PERIODS, "seed": 0})
ft_env_smoke = make_env({"fixed_demand": True, "num_periods": NUM_PERIODS, "seed": 0})

obs, info = ft_env_smoke.reset(seed=0)
obs2, reward, terminated, truncated, info2 = ft_env_smoke.step(ft_env_smoke.action_space.sample())

print("Train env:", type(train_env_smoke).__name__)
print("FT env:", type(ft_env_smoke).__name__)
print("action_space:", ft_env_smoke.action_space)
print("observation_space:", ft_env_smoke.observation_space)
print("first reward:", reward)
print("mo_reward:", info2.get("mo_reward"))
print("mo_reward_raw:", info2.get("mo_reward_raw"))

Train env: SimpleSupplyChainEnv
FT env: FTSimpleSupplyChainEnv
action_space: Box(0.0, 500.0, (8,), float32)
observation_space: Box(-inf, inf, (24,), float32)
first reward: -0.10363119179586248
mo_reward: [ 0.20211564 -0.01300921 -0.5       ]
mo_reward_raw: [ 2.4819800e+03 -2.6083768e+02 -5.0000000e-01]


In [26]:
# ---------------------------------------------------------------------
# 3. RLlib environment registration
# ---------------------------------------------------------------------
TRAIN_ENV_ID = "miracl_simple_train_orgym"
FT_ENV_ID = "miracl_simple_ft_orgym"


def make_train_env(env_config=None):
    cfg = dict(env_config or {})
    cfg.setdefault("fixed_demand", False)
    cfg.setdefault("num_periods", NUM_PERIODS)
    env = make_env(cfg)
    return TimeLimit(env, max_episode_steps=cfg["num_periods"])


def make_ft_env(env_config=None):
    cfg = dict(env_config or {})
    cfg.setdefault("fixed_demand", True)
    cfg.setdefault("num_periods", NUM_PERIODS)
    cfg.setdefault("randomise_costs", False)
    cfg.setdefault("randomise_demand", False)
    env = make_env(cfg)
    return TimeLimit(env, max_episode_steps=cfg["num_periods"])

register_env(TRAIN_ENV_ID, make_train_env)
register_env(FT_ENV_ID, make_ft_env)

print("Registered RLlib envs:", TRAIN_ENV_ID, FT_ENV_ID)

Registered RLlib envs: miracl_simple_train_orgym miracl_simple_ft_orgym


## Meta-training phase

Set `RUN_META_TRAINING=True` to run full training.  The default is `False` so the notebook can be opened and smoke-tested without launching a long Ray job.

In [27]:
# ---------------------------------------------------------------------
# 4. MAML/PSA meta-training config
# ---------------------------------------------------------------------
ray.shutdown()
ray.init(ignore_reinit_error=True)

meta_config = (
    MAMLConfigPsa()
    .rollouts(num_rollout_workers=4, rollout_fragment_length=ROLLOUT_FRAGMENT_LENGTH)
    .framework("torch")
    .environment(TRAIN_ENV_ID, clip_actions=True, env_config={"fixed_demand": False, "num_periods": NUM_PERIODS})
    .training(
        inner_adaptation_steps=4,
        maml_optimizer_steps=10,
        gamma=0.99,
        lambda_=1.0,
        lr=0.001,
        vf_loss_coeff=0.5,
        inner_lr=0.003,   # paper setting; change to 0.03 only if restoring old checkpoints trained that way
        use_meta_env=True,
        clip_param=0.3,
        kl_target=0.01,
        kl_coeff=0.001,
        model=dict(fcnet_hiddens=[64, 64], fcnet_activation="tanh"),
        train_batch_size=ROLLOUT_FRAGMENT_LENGTH,
    )
    .evaluation(
        evaluation_num_workers=1,
        evaluation_interval=100,
        enable_async_evaluation=True,
    )
)

meta_config.to_dict()["env"]

2026-05-07 02:29:55,268	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-05-07 02:29:55,269	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-05-07 02:29:55,271	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67104768 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

'miracl_simple_train_orgym'

In [28]:
# ---------------------------------------------------------------------
# 5. Run meta-training and save one checkpoint per run
# ---------------------------------------------------------------------
RUN_META_TRAINING = False
NUM_ITERATIONS = 250  # 250 * 4 workers * 32 fragment length is a small run; adjust for your target budget.

meta_results = {}

if RUN_META_TRAINING:
    for run_id in range(RUNS):
        print(f"RUNNING META-TRAINING {run_id}")
        tuner = tune.Tuner(
            MAMLPsa,
            param_space=meta_config.to_dict(),
            run_config=air.RunConfig(
                stop={"training_iteration": NUM_ITERATIONS},
                failure_config=air.FailureConfig(fail_fast="raise"),
                log_to_file=True,
            ),
        )
        result_grid = tuner.fit()
        meta_results[run_id] = result_grid

        best_result = result_grid.get_best_result()
        run_ckpt = CKPT_DIR / f"run_{run_id}_metapolicy"
        run_ckpt.mkdir(parents=True, exist_ok=True)
        best_result.checkpoint.to_directory(str(run_ckpt))
        print(f"Best checkpoint for run {run_id} stored in: {run_ckpt}")
else:
    print("RUN_META_TRAINING=False; skipping meta-training.")

RUN_META_TRAINING=False; skipping meta-training.


## Fine-tuning phase with Stable-Baselines3 PPO

This section restores each RLlib meta-policy checkpoint (if available), starts an SB3 PPO policy on the OR-Gym-style fixed-demand environment, and evaluates vector returns from `simple_state.py`.

In [29]:
# ---------------------------------------------------------------------
# 6. Utilities: weights, evaluation, and optional RLlib -> SB3 warm start
# ---------------------------------------------------------------------
WEIGHT_GRID_21 = np.array([
    [0.0, 0.0, 1.0], [0.0, 0.2, 0.8], [0.0, 0.4, 0.6], [0.0, 0.6, 0.4], [0.0, 0.8, 0.2],
    [0.0, 1.0, 0.0], [0.2, 0.0, 0.8], [0.2, 0.2, 0.6], [0.2, 0.4, 0.4], [0.2, 0.6, 0.2],
    [0.2, 0.8, 0.0], [0.4, 0.0, 0.6], [0.4, 0.2, 0.4], [0.4, 0.4, 0.2], [0.4, 0.6, 0.0],
    [0.6, 0.0, 0.4], [0.6, 0.2, 0.2], [0.6, 0.4, 0.0], [0.8, 0.0, 0.2], [0.8, 0.2, 0.0],
    [1.0, 0.0, 0.0],
], dtype=np.float64)


def set_env_weights(env, weights):
    """Set scalarisation weights through Monitor/TimeLimit wrappers if needed."""
    target = env
    while hasattr(target, "env"):
        if hasattr(target, "set_scalarization_weights"):
            break
        target = target.env
    if hasattr(target, "set_scalarization_weights"):
        target.set_scalarization_weights(weights)
    else:
        raise AttributeError("Could not find set_scalarization_weights on env or wrapped env.")


def make_sb3_env(weights, seed=None):
    env = make_env({
        "fixed_demand": True,
        "num_periods": NUM_PERIODS,
        "weights": np.asarray(weights, dtype=np.float64),
        "seed": seed,
        "normalize_actions": True,      # important
        "max_order_quantity": 500.0,
    })
    return Monitor(env)


def evaluate_model(model, weights, n_eval_episodes=50, seed=123):
    """Evaluate an SB3 model and return scalar and vector-return statistics.

    Returns raw vector returns in the convention of simple_state.py:
    [profit, -emission, -service_inequality].
    """
    scalar_returns = []
    raw_returns = []
    norm_returns = []

    for ep in range(n_eval_episodes):
        env = make_sb3_env(weights, seed=None if seed is None else seed + ep)
        obs, info = env.reset(seed=None if seed is None else seed + ep)
        done = False
        scalar_total = 0.0
        raw_total = np.zeros(3, dtype=np.float64)
        norm_total = np.zeros(3, dtype=np.float64)

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = bool(terminated or truncated)
            scalar_total += float(reward)
            raw_total += np.asarray(info.get("mo_reward_raw", np.zeros(3)), dtype=np.float64)
            norm_total += np.asarray(info.get("mo_reward", np.zeros(3)), dtype=np.float64)

        scalar_returns.append(scalar_total)
        raw_returns.append(raw_total)
        norm_returns.append(norm_total)
        env.close()

    scalar_returns = np.asarray(scalar_returns, dtype=np.float64)
    raw_returns = np.asarray(raw_returns, dtype=np.float64)
    norm_returns = np.asarray(norm_returns, dtype=np.float64)

    return {
        "mean_scalar": float(scalar_returns.mean()),
        "std_scalar": float(scalar_returns.std(ddof=1)) if len(scalar_returns) > 1 else 0.0,
        "mean_raw_vec": raw_returns.mean(axis=0),
        "std_raw_vec": raw_returns.std(axis=0, ddof=1) if len(raw_returns) > 1 else np.zeros(3),
        "mean_norm_vec": norm_returns.mean(axis=0),
        "std_norm_vec": norm_returns.std(axis=0, ddof=1) if len(norm_returns) > 1 else np.zeros(3),
    }


def build_restored_maml_trainer(checkpoint_path):
    """Build and restore the RLlib MAML trainer for a saved checkpoint."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint path does not exist: {checkpoint_path}")

    trainer = meta_config.build()
    trainer.restore(str(checkpoint_path))
    return trainer


def warm_start_sb3_from_rllib(sb3_model, rllib_trainer):
    """Best-effort copy of matching torch parameters from RLlib to SB3.

    RLlib and SB3 often use different module names, so this function only copies
    parameters with identical keys and shapes. If nothing matches, training still
    runs from the SB3 initialisation and a warning is printed.
    """
    if rllib_trainer is None:
        return 0
    try:
        ray_state = rllib_trainer.get_policy().model.state_dict()
        sb3_state = sb3_model.policy.state_dict()
        copied = 0
        for key, value in ray_state.items():
            if key in sb3_state and tuple(value.shape) == tuple(sb3_state[key].shape):
                sb3_state[key] = value.detach().cpu()
                copied += 1
        if copied:
            sb3_model.policy.load_state_dict(sb3_state)
        else:
            warnings.warn("No matching RLlib/SB3 parameter names found; SB3 policy starts from its own initialisation.")
        return copied
    except Exception as exc:
        warnings.warn(f"Could not warm-start SB3 from RLlib checkpoint: {exc}")
        return 0

In [ ]:
# ---------------------------------------------------------------------
# 7. Fine-tune one or more checkpoints with SB3 PPO
# ---------------------------------------------------------------------
RUN_FINE_TUNING = True
FT_TOTAL_TIMESTEPS = 5000
N_SEEDS = 3
WEIGHT_GRID = WEIGHT_GRID_21[:3]
USE_RLLIB_WARM_START = False

n_records = int(TOTAL_STEPS / RECORD_EVERY)
episode_num = np.linspace(RECORD_EVERY, TOTAL_STEPS, n_records, dtype=int)

if RUN_FINE_TUNING:
    for run_id in range(RUNS):
        ray.shutdown()
        ray.init(ignore_reinit_error=True)
        register_env(TRAIN_ENV_ID, make_train_env)
        register_env(FT_ENV_ID, make_ft_env)

        ckpt_path = CKPT_DIR / f"run_{run_id}_metapolicy"
        rllib_trainer = None
        if USE_RLLIB_WARM_START and ckpt_path.exists():
            print(f"Restoring RLlib checkpoint: {ckpt_path}")
            rllib_trainer = build_restored_maml_trainer(ckpt_path)
        elif USE_RLLIB_WARM_START:
            warnings.warn(f"Checkpoint not found for run {run_id}: {ckpt_path}. Running SB3 from scratch.")

        print(f"RUNNING FINE-TUNING RUN {run_id}")
        start = time.time()

        mean_reward_history = np.zeros((WEIGHT_GRID_21.shape[0], n_records))
        std_reward_history = np.zeros((WEIGHT_GRID_21.shape[0], n_records))
        mean_profit_hist = np.zeros((WEIGHT_GRID_21.shape[0], n_records))
        mean_emission_hist = np.zeros((WEIGHT_GRID_21.shape[0], n_records))
        mean_equity_hist = np.zeros((WEIGHT_GRID_21.shape[0], n_records))

        for weight_id, weights in enumerate(WEIGHT_GRID_21):
            print(f"start training solution {weight_id}, weights={weights}")
            env = make_sb3_env(weights, seed=run_id)
            model = PPO(
                MlpPolicy,
                env,
                verbose=0,
                learning_rate=3e-4,
                n_steps=2048,
                batch_size=64,
                n_epochs=10,
                gamma=0.99,
                gae_lambda=0.95,
                clip_range=0.2,
                ent_coef=0.0,
                vf_coef=0.5,
                max_grad_norm=0.5,
                policy_kwargs=dict(net_arch=[64, 64]),
                seed=run_id,
            )
            copied = warm_start_sb3_from_rllib(model, rllib_trainer)
            if copied:
                print(f"Copied {copied} matching tensors from RLlib to SB3")

            for rec_id in range(n_records):
                model.learn(total_timesteps=RECORD_EVERY, reset_num_timesteps=False)
                stats = evaluate_model(model, weights, n_eval_episodes=EVAL_EPISODES, seed=10_000 + run_id * 1000 + weight_id * 10)

                mean_reward_history[weight_id, rec_id] = stats["mean_scalar"]
                std_reward_history[weight_id, rec_id] = stats["std_scalar"]
                mean_profit_hist[weight_id, rec_id] = stats["mean_raw_vec"][0]
                mean_emission_hist[weight_id, rec_id] = stats["mean_raw_vec"][1]
                mean_equity_hist[weight_id, rec_id] = stats["mean_raw_vec"][2]

                print(
                    f"  step={episode_num[rec_id]} scalar={stats['mean_scalar']:.3f} "
                    f"vec={stats['mean_raw_vec']}"
                )

            model_file = FT_DIR / f"run_{run_id}_solution_{weight_id}.zip"
            model.save(str(model_file))
            env.close()

        print(f"Training time for run {run_id}: {time.time() - start:.1f} seconds")
        print("=" * 50)

        pd.DataFrame(WEIGHT_GRID_21).to_csv(FT_DIR / f"{run_id}.weight_list.csv", index=False)
        pd.DataFrame(mean_reward_history).to_csv(FT_DIR / f"{run_id}.mean_reward.csv", index=False)
        pd.DataFrame(std_reward_history).to_csv(FT_DIR / f"{run_id}.std_reward.csv", index=False)
        pd.DataFrame(mean_profit_hist).to_csv(FT_DIR / f"{run_id}.profit_reward.csv", index=False)
        pd.DataFrame(mean_emission_hist).to_csv(FT_DIR / f"{run_id}.emission_reward.csv", index=False)
        pd.DataFrame(mean_equity_hist).to_csv(FT_DIR / f"{run_id}.equity_reward.csv", index=False)

    print("ALL RUNS DONE")
else:
    print("RUN_FINE_TUNING=False; skipping SB3 fine-tuning.")

2026-05-07 02:30:00,827	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-05-07 02:30:00,829	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-05-07 02:30:00,830	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67104768 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

RUNNING FINE-TUNING RUN 0
start training solution 0, weights=[0. 0. 1.]
  step=250 scalar=-2.435 vec=[-1.76346698e+04 -1.73186445e+03 -2.43528532e+00]
  step=500 scalar=-4.502 vec=[-2.06529751e+04 -1.09756904e+04 -4.50172780e+00]
  step=750 scalar=-3.686 vec=[-4.40880956e+04 -1.46140563e+04 -3.68588987e+00]
  step=1000 scalar=-7.916 vec=[ 9.16397497e+03 -1.84950147e+04 -7.91555751e+00]
  step=1250 scalar=-8.341 vec=[ 1.19099496e+03 -2.36398130e+04 -8.34089487e+00]
  step=1500 scalar=-14.085 vec=[ 1.56160502e+04 -3.07904511e+04 -1.40851481e+01]
  step=1750 scalar=-12.369 vec=[-1.04864053e+04 -3.52294987e+04 -1.23690970e+01]
  step=2000 scalar=-13.867 vec=[-1.27724295e+04 -4.30091716e+04 -1.38669209e+01]
  step=2250 scalar=-12.012 vec=[-1.73875046e+03 -3.67619713e+04 -1.20117596e+01]
  step=2500 scalar=-10.186 vec=[-2.27053796e+04 -4.74932765e+04 -1.01863897e+01]
  step=2750 scalar=-12.059 vec=[-2.61399501e+04 -5.93766414e+04 -1.20585411e+01]
  step=3000 scalar=-13.526 vec=[ 3.83139198e+

## Reload saved fine-tuning CSVs

In [ ]:
# ---------------------------------------------------------------------
# 8. Reload fine-tuning results from CSV
# ---------------------------------------------------------------------
def load_fine_tuning_results(ft_dir=FT_DIR, runs=RUNS):
    weight_list, mean_reward, std_reward = [], [], []
    profit_reward, emission_reward, equity_reward = [], [], []

    for run_id in range(runs):
        weight_list.append(pd.read_csv(ft_dir / f"{run_id}.weight_list.csv").to_numpy())
        mean_reward.append(pd.read_csv(ft_dir / f"{run_id}.mean_reward.csv").to_numpy())
        std_reward.append(pd.read_csv(ft_dir / f"{run_id}.std_reward.csv").to_numpy())
        profit_reward.append(pd.read_csv(ft_dir / f"{run_id}.profit_reward.csv").to_numpy())
        emission_reward.append(pd.read_csv(ft_dir / f"{run_id}.emission_reward.csv").to_numpy())
        equity_reward.append(pd.read_csv(ft_dir / f"{run_id}.equity_reward.csv").to_numpy())

    return {
        "weight_list": np.asarray(weight_list),
        "mean_reward": np.asarray(mean_reward),
        "std_reward": np.asarray(std_reward),
        "profit_reward": np.asarray(profit_reward),
        "emission_reward": np.asarray(emission_reward),
        "equity_reward": np.asarray(equity_reward),
    }

# Uncomment after running fine-tuning.
# data = load_fine_tuning_results()
# weight_list = data["weight_list"]
# mean_reward = data["mean_reward"]
# std_reward = data["std_reward"]
# profit_reward = data["profit_reward"]
# emission_reward = data["emission_reward"]
# equity_reward = data["equity_reward"]

## Performance metrics

In [ ]:
# ---------------------------------------------------------------------
# 9. Hypervolume calculation
# ---------------------------------------------------------------------
def compute_hypervolume_curves(profit_reward, emission_reward, equity_reward):
    if pg is None:
        raise ImportError("pygmo is required for hypervolume calculation")

    min_point = np.array([0.0, -2e5, -1e2], dtype=np.float64)
    max_point = np.array([4e5, 0.0, 0.0], dtype=np.float64)
    ref_point = np.array([4e5, 0.0, 0.0], dtype=np.float64)

    reversed_profit = max_point[0] - (profit_reward - min_point[0])
    reversed_emission = max_point[1] - (emission_reward - min_point[1])
    reversed_equity = max_point[2] - (equity_reward - min_point[2])
    reversed_rewards = np.stack([reversed_profit, reversed_emission, reversed_equity], axis=-1)

    final_h = []
    for run_id in range(reversed_rewards.shape[0]):
        h = []
        for rec_id in range(reversed_rewards.shape[2]):
            points = np.minimum(reversed_rewards[run_id, :, rec_id, :], ref_point - 1e-5)
            hv = pg.hypervolume(points)
            h.append(hv.compute(ref_point))
        final_h.append(h)

    n_runs = len(final_h)
    df_h = pd.DataFrame(final_h).T
    df_h["mean"] = df_h.mean(axis=1)
    df_h["std"] = df_h.std(axis=1)
    df_h["upper"] = df_h["mean"] + 1.96 * df_h["std"] / np.sqrt(n_runs)
    df_h["lower"] = df_h["mean"] - 1.96 * df_h["std"] / np.sqrt(n_runs)
    return df_h

# Example after loading CSVs:
# df_h = compute_hypervolume_curves(profit_reward, emission_reward, equity_reward)
# df_h.to_csv(PERF_DIR / "hypervolume.csv", index=False)
# df_h.head()

In [ ]:
# ---------------------------------------------------------------------
# 10. Sparsity calculation
# ---------------------------------------------------------------------
def calculate_sparsity_3d(pareto_front_3d):
    pareto_front = np.asarray(pareto_front_3d, dtype=np.float64).reshape(-1, pareto_front_3d.shape[-1])
    sorted_front = pareto_front[pareto_front[:, 0].argsort()]
    if len(sorted_front) < 2:
        return 0.0
    squared_distances = np.sum(np.diff(sorted_front, axis=0) ** 2, axis=1)
    return float(np.mean(squared_distances))


def compute_sparsity_curves(profit_reward, emission_reward, equity_reward):
    combined_rewards = np.stack([profit_reward, emission_reward, equity_reward], axis=-1)
    final_s = []
    for run_id in range(combined_rewards.shape[0]):
        s = []
        for rec_id in range(combined_rewards.shape[2]):
            s.append(calculate_sparsity_3d(combined_rewards[run_id, :, rec_id, :]))
        final_s.append(s)
    n_runs = len(final_s)
    df_s = pd.DataFrame(final_s).T
    df_s["mean"] = df_s.mean(axis=1)
    df_s["std"] = df_s.std(axis=1)
    df_s["upper"] = df_s["mean"] + 1.96 * df_s["std"] / np.sqrt(n_runs)
    df_s["lower"] = df_s["mean"] - 1.96 * df_s["std"] / np.sqrt(n_runs)
    return df_s

# df_s = compute_sparsity_curves(profit_reward, emission_reward, equity_reward)
# df_s.to_csv(PERF_DIR / "sparsity.csv", index=False)
# df_s.head()

In [ ]:
# ---------------------------------------------------------------------
# 11. EUM calculation
# ---------------------------------------------------------------------
def compute_eum_curves(weight_list, profit_reward, emission_reward, equity_reward):
    combined_rewards = np.stack([profit_reward, emission_reward, equity_reward], axis=-1)
    final_e = []
    for run_id in range(combined_rewards.shape[0]):
        e = []
        for rec_id in range(combined_rewards.shape[2]):
            utilities = []
            for weight_id in range(combined_rewards.shape[1]):
                reward_vector = combined_rewards[run_id, weight_id, rec_id]
                weights = weight_list[run_id, weight_id]
                utilities.append(float(np.dot(reward_vector, weights)))
            e.append(float(np.mean(utilities)))
        final_e.append(e)
    n_runs = len(final_e)
    df_e = pd.DataFrame(final_e).T
    df_e["mean"] = df_e.mean(axis=1)
    df_e["std"] = df_e.std(axis=1)
    df_e["upper"] = df_e["mean"] + 1.96 * df_e["std"] / np.sqrt(n_runs)
    df_e["lower"] = df_e["mean"] - 1.96 * df_e["std"] / np.sqrt(n_runs)
    return df_e

# df_e = compute_eum_curves(weight_list, profit_reward, emission_reward, equity_reward)
# df_e.to_csv(PERF_DIR / "eum.csv", index=False)
# df_e.head()

In [ ]:
# ---------------------------------------------------------------------
# 12. Plot metric curves
# ---------------------------------------------------------------------
def plot_curve(df, ylabel, total_steps=TOTAL_STEPS, record_every=RECORD_EVERY):
    x = np.arange(record_every, total_steps + 1, record_every)
    plt.figure(figsize=(6, 4))
    plt.plot(x, df["mean"])
    plt.fill_between(x, df["lower"], df["upper"], alpha=0.15)
    plt.xlabel("timesteps", fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.tight_layout()
    plt.show()

# plot_curve(df_h, "hypervolume")
# plot_curve(df_s, "sparsity")
# plot_curve(df_e, "EUM")